In [ ]:
!nvidia-smi

In [ ]:
%%writefile vector_add.cu
#include <stdio.h>

__global__ void add(int *a, int *b, int *c) {
    int id = threadIdx.x;
    c[id] = a[id] + b[id];
}

int main() {
    int a[5] = {1,2,3,4,5};
    int b[5] = {5,4,3,2,1};
    int c[5];

    int *d_a, *d_b, *d_c;

    cudaMalloc(&d_a, 5*sizeof(int));
    cudaMalloc(&d_b, 5*sizeof(int));
    cudaMalloc(&d_c, 5*sizeof(int));

    cudaMemcpy(d_a, a, 5*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, 5*sizeof(int), cudaMemcpyHostToDevice);

    add<<<1,5>>>(d_a, d_b, d_c);

    cudaMemcpy(c, d_c, 5*sizeof(int), cudaMemcpyDeviceToHost);

    for(int i=0;i<5;i++)
        printf("%d ", c[i]);

    return 0;
}



!nvcc vector_add.cu -o vector_add

!./vector_add




%%writefile matrix_mul.cu
#include <stdio.h>

__global__ void mul(int *A, int *B, int *C) {
    int row = threadIdx.y;
    int col = threadIdx.x;
    int sum = 0;

    for(int k=0;k<2;k++)
        sum += A[row*2+k]*B[k*2+col];

    C[row*2+col] = sum;
}

int main() {
    int A[4] = {1,2,3,4};
    int B[4] = {5,6,7,8};
    int C[4];

    int *d_A,*d_B,*d_C;

    cudaMalloc(&d_A,4*sizeof(int));
    cudaMalloc(&d_B,4*sizeof(int));
    cudaMalloc(&d_C,4*sizeof(int));

    cudaMemcpy(d_A,A,4*sizeof(int),cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,B,4*sizeof(int),cudaMemcpyHostToDevice);

    dim3 threads(2,2);
    mul<<<1,threads>>>(d_A,d_B,d_C);

    cudaMemcpy(C,d_C,4*sizeof(int),cudaMemcpyDeviceToHost);

    for(int i=0;i<4;i++)
        printf("%d ",C[i]);

    return 0;
}





In [ ]:

!nvcc matrix_mul.cu -o matrix_mul
!./matrix_mul